# Simulação NAN — Número de Atendentes Necessários

Simula a operação de um call center ao longo de um dia, determinando o **número mínimo de atendentes** necessários para atender todas as ligações sem espera excessiva.

**Fluxo geral:**
1. Carrega dados reais de ligantes e durações de conexão
2. Ajusta um polinômio para modelar a distribuição de durações
3. Gera momentos de ligação aleatórios
4. Simula a atribuição de ligações aos atendentes disponíveis

In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

np.set_printoptions(legacy="1.13")

## Carregamento dos Dados

Lê as duas tabelas de entrada:
- `df_tabela_ligantes` — número médio de ligantes por faixa horária
- `df_duracoes` — distribuição acumulada da duração das conexões

In [83]:
df_tabela_ligantes = pd.read_excel("df_tabela_ligantes (2).xlsx")
display(df_tabela_ligantes)

,Rótulo,Hora Início,Hora Fim,Número Médio Ligantes
0,1,00:00:00,00:15:00,601
1,2,00:15:00,00:30:00,190
2,3,00:30:00,00:45:00,180
3,4,00:45:00,01:00:00,177
4,5,01:00:00,01:15:00,162
...,...,...,...,...
91,92,22:45:00,23:00:00,920
92,93,23:00:00,23:15:00,897
93,94,23:15:00,23:30:00,571
94,95,23:30:00,23:45:00,560


In [84]:
df_duracoes = pd.read_excel("df_duracoes.xlsx")
display(df_duracoes)

,Duração Conexão,Número Ligantes,Percentagem Relativa,Percentagem Acumulada
0,0,0,0.000,0.000
1,5,870,0.087,0.087
2,10,2210,0.221,0.308
3,15,2940,0.294,0.602
4,20,1710,0.171,0.773
5,25,1360,0.136,0.909
6,30,590,0.059,0.968
7,35,250,0.025,0.993
8,40,70,0.007,1.000


## Modelagem da Duração das Ligações

Ajusta um polinômio de grau 5 à curva acumulada de duração, permitindo gerar durações aleatórias realistas via `ft(p)`, onde `p` é um valor uniforme em [0, 1].

In [85]:
coefs = np.polyfit(
    x=df_duracoes["Percentagem Acumulada"],
    y=df_duracoes["Duração Conexão"],
    deg=5
)
ft = np.poly1d(coefs, variable="p")
print(ft)

       5        4        3         2
725.3 p - 1678 p + 1413 p - 523.8 p + 101 p - 0.2063


In [86]:
# para p = 0 => t = -0.20s
print(ft(0))

-0.206253859128


In [87]:
# para p = 1 => t = 37.63s
print(ft(1))

37.6312985354


In [88]:
qtd_ligantes = df_tabela_ligantes.loc[0, "Número Médio Ligantes"]
print(qtd_ligantes)

601


## Simulação Passo a Passo

Para entender a lógica antes de automatizar, simulamos manualmente as primeiras ligações da primeira faixa horária. Cada atendente é representado por um dicionário com `inicio` e `final` (em segundos).

> **Regra:** um atendente é considerado **ocupado** se este desocupará após `10 segundos` do momento da chegada da ligação

In [89]:
# sorteio dos momentos (segundos) das ligação que serão simuladas
np.random.seed(40)
momentos_ligacoes = np.random.randint(low=0, high=900,
                                      size=qtd_ligantes)
momentos_ligacoes.sort()
print(momentos_ligacoes)

[  5   7   7   8   8  10  11  17  21  22  22  23  24  24  24  25  25  29
  32  33  35  36  37  38  38  40  44  45  47  52  55  57  57  57  58  59
  59  59  60  60  61  63  65  67  67  68  69  72  73  76  80  81  81  83
  87  88  89  89  89  89  90  91  94  94  94  97  97  99 101 103 105 105
 107 108 109 115 115 117 119 121 123 123 123 124 125 126 127 130 131 134
 134 134 138 142 142 153 156 156 156 156 160 162 162 165 171 173 174 175
 175 178 179 180 180 180 182 184 187 189 189 190 191 201 205 206 207 207
 207 207 208 210 210 211 211 211 214 219 219 219 224 226 227 228 228 229
 231 234 235 241 241 241 241 241 242 244 245 247 248 249 249 250 250 250
 254 255 255 257 258 258 258 259 261 263 268 268 270 270 270 273 276 278
 278 280 286 288 289 289 294 295 296 297 297 300 300 302 302 305 306 306
 306 308 310 311 311 315 317 317 317 320 321 322 324 325 326 326 327 329
 330 338 338 343 344 347 347 349 349 351 354 355 357 357 357 359 362 364
 372 375 377 378 384 384 384 387 389 391 391 394 39

In [90]:
# inicializando (antes de simular 00:00 - 00:15) a lista de atendentes vazia
atendentes: list[dict] = []

---
### (Primeiro período simulado) `00:00` - `00:15`

In [91]:
momento_ligacao = momentos_ligacoes[0]
print(momento_ligacao)

5


In [92]:
np.random.seed(43)
duracao = ft(np.random.random())
print(duracao)

6.3545805175


In [93]:
if len(atendentes) == 0:
    atendente = {"inicio": momento_ligacao,
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display(atendentes)

[{'inicio': 5, 'final': 11.354580517497224}]

---
### (Segundo período simulado) `00:15` - `00:30`

In [94]:
momento_ligacao = momentos_ligacoes[1]
print(momento_ligacao)

7


In [95]:
np.random.seed(44)
duracao = ft(np.random.random())
print(duracao)

20.4274112657


In [96]:
display(atendentes)

[{'inicio': 5, 'final': 11.354580517497224}]

**Lógica de atribuição:** percorre a lista de atendentes procurando um que esteja livre dentro do intervalo `<momento_ligacao> + 10`. Se nenhum estiver disponível (cláusula `else` do `for`), cria um novo atendente. Se algum atendente estiver livre, atualiza seu `inicio` e `final` de acordo com a regra: 
#### `Se o atendente estiver desocupado no momento da ligação, ele vai atender imediatamente, caso contrário, atende quando desocupar`.

In [97]:
for atendente in atendentes:
    ocupado = atendente["final"] > momento_ligacao + 10
    if not ocupado:
        # atualizar atendente
        if atendente["final"] < momento_ligacao:
            # atendente já desocupado
            atendente["inicio"] = momento_ligacao
        else:
            # atendente vai desocupar nos próximos momentos
            atendente["inicio"] = atendente["final"]

        atendente["final"] = atendente["inicio"] + duracao
        break
else:
    atendente = {"inicio": momento_ligacao, 
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display(atendentes)

[{'inicio': 11.354580517497224, 'final': 31.781991783170163}]

---
### (Terceiro período simulado) `00:30` - `00:45`

In [98]:
momento_ligacao = momentos_ligacoes[2]
np.random.seed(45)
duracao = ft(np.random.random())
print(f"NOVA LIGAÇÃO: momento ligacao: {momento_ligacao} | duracao_ligacao: {duracao}")
print("-" * 100)
display("atendentes", atendentes)

NOVA LIGAÇÃO: momento ligacao: 7 | duracao_ligacao: 35.452411285016005
----------------------------------------------------------------------------------------------------


'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163}]

In [99]:
for atendente in atendentes:
    ocupado = atendente["final"] > momento_ligacao + 10
    if not ocupado:
        # atualizar atendente
        if atendente["final"] < momento_ligacao:
            # atendente já desocupado
            atendente["inicio"] = momento_ligacao
        else:
            # atendente vai desocupar nos próximos momentos
            atendente["inicio"] = atendente["final"]

        atendente["final"] = atendente["inicio"] + duracao
        break
else:
    atendente = {"inicio": momento_ligacao, 
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display(atendentes)

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005}]

---
### (Quarto período simulado) `00:45` - `01:00`

In [100]:
momento_ligacao = momentos_ligacoes[3]
np.random.seed(46)
duracao = ft(np.random.random())
print(f"NOVA LIGAÇÃO: momento ligacao: {momento_ligacao} | duracao_ligacao: {duracao}")
print("-" * 100)
display("atendentes", atendentes)

NOVA LIGAÇÃO: momento ligacao: 8 | duracao_ligacao: 18.966305047480674
----------------------------------------------------------------------------------------------------


'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005}]

In [101]:
for atendente in atendentes:
    ocupado = atendente["final"] > momento_ligacao + 10
    if not ocupado:
        # atualizar atendente
        if atendente["final"] < momento_ligacao:
            # atendente já desocupado
            atendente["inicio"] = momento_ligacao
        else:
            # atendente vai desocupar nos próximos momentos
            atendente["inicio"] = atendente["final"]

        atendente["final"] = atendente["inicio"] + duracao
        break
else:
    atendente = {"inicio": momento_ligacao, 
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display(atendentes)

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005},
 {'inicio': 8, 'final': 26.966305047480674}]

---
### (Quinto período simulado) `01:00` - `01:15`

In [102]:
momento_ligacao = momentos_ligacoes[4]
np.random.seed(47)
duracao = ft(np.random.random())
print(f"NOVA LIGAÇÃO: momento ligacao: {momento_ligacao} | duracao_ligacao: {duracao}")
print("-" * 100)
display("atendentes", atendentes)

NOVA LIGAÇÃO: momento ligacao: 8 | duracao_ligacao: 6.3118813445620665
----------------------------------------------------------------------------------------------------


'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005},
 {'inicio': 8, 'final': 26.966305047480674}]

In [103]:
for atendente in atendentes:
    ocupado = atendente["final"] > momento_ligacao + 10
    if not ocupado:
        # atualizar atendente
        if atendente["final"] < momento_ligacao:
            # atendente já desocupado
            atendente["inicio"] = momento_ligacao
        else:
            # atendente vai desocupar nos próximos momentos
            atendente["inicio"] = atendente["final"]

        atendente["final"] = atendente["inicio"] + duracao
        break
else:
    atendente = {"inicio": momento_ligacao, 
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display("atendentes", atendentes)

'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005},
 {'inicio': 8, 'final': 26.966305047480674},
 {'inicio': 8, 'final': 14.311881344562067}]

---
### (Sexto período simulado) `01:15` - `01:30`

In [104]:
momento_ligacao = momentos_ligacoes[5]
np.random.seed(48)
duracao = ft(np.random.random())
print(f"NOVA LIGAÇÃO: momento ligacao: {momento_ligacao} | duracao_ligacao: {duracao}")
print("-" * 100)
display("atendentes", atendentes)

NOVA LIGAÇÃO: momento ligacao: 10 | duracao_ligacao: 1.4076159104542545
----------------------------------------------------------------------------------------------------


'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005},
 {'inicio': 8, 'final': 26.966305047480674},
 {'inicio': 8, 'final': 14.311881344562067}]

In [105]:
for atendente in atendentes:
    ocupado = atendente["final"] > momento_ligacao + 10
    if not ocupado:
        # atualizar atendente
        if atendente["final"] < momento_ligacao:
            # atendente já desocupado
            atendente["inicio"] = momento_ligacao
        else:
            # atendente vai desocupar nos próximos momentos
            atendente["inicio"] = atendente["final"]

        atendente["final"] = atendente["inicio"] + duracao
        break
else:
    atendente = {"inicio": momento_ligacao, 
                 "final": momento_ligacao + duracao}
    atendentes.append(atendente)

display("atendentes", atendentes)

'atendentes'

[{'inicio': 11.354580517497224, 'final': 31.781991783170163},
 {'inicio': 7, 'final': 42.452411285016005},
 {'inicio': 8, 'final': 26.966305047480674},
 {'inicio': 14.311881344562067, 'final': 15.719497255016321}]

---
### Juntando a lógica para realizar `TODOS` os NANs

In [106]:
np.random.seed(1)
coluna_nan = []
espera = 10

for qtd_ligantes in df_tabela_ligantes["Número Médio Ligantes"]:
    # gera os momentos de ligação (em segundos dentro da janela de 15 min = 900s)
    momentos_ligacoes = np.random.randint(low=0, high=900, size=qtd_ligantes)
    momentos_ligacoes.sort()

    # cria uma lista de atendentes vazia para cada nova simulacão
    atendentes = []  # lista de atendentes da faixa horária atual

    for momento_ligacao in momentos_ligacoes:
        duracao = ft(np.random.random())  # duração sorteada pelo polinômio ajustado

        # primeiro atendente da faixa: sempre cria um novo
        if len(atendentes) == 0:
            atendentes.append({"inicio": momento_ligacao,
                               "final": momento_ligacao + duracao})
            continue

        for atendente in atendentes:
            ocupado = atendente["final"] > momento_ligacao + 10
            if not ocupado:
                # atualizar atendente
                if atendente["final"] < momento_ligacao:
                    # atendente já desocupado
                    atendente["inicio"] = momento_ligacao
                else:
                    # atendente vai desocupar nos próximos momentos
                    atendente["inicio"] = atendente["final"]

                atendente["final"] = atendente["inicio"] + duracao
                break
        else:
            atendente = {"inicio": momento_ligacao, 
                        "final": momento_ligacao + duracao}
            atendentes.append(atendente)

    nan = len(atendentes)  # NAN = número de atendentes necessários na faixa
    coluna_nan.append(nan)

print(coluna_nan)

[14, 7, 8, 7, 6, 6, 5, 4, 6, 4, 4, 4, 5, 2, 3, 4, 3, 3, 3, 4, 3, 3, 4, 3, 4, 4, 6, 6, 15, 15, 20, 19, 36, 33, 33, 35, 55, 55, 54, 62, 65, 66, 59, 63, 58, 57, 55, 65, 47, 54, 56, 55, 55, 66, 57, 60, 56, 57, 63, 54, 53, 63, 62, 60, 60, 56, 61, 65, 69, 68, 62, 56, 49, 40, 44, 40, 38, 38, 34, 35, 35, 31, 34, 36, 33, 26, 23, 24, 26, 24, 21, 23, 21, 15, 14, 18]


In [107]:
# adicionando uma coluna no dataframe
df_tabela_ligantes["nan"] = coluna_nan
df_tabela_ligantes

,Rótulo,Hora Início,Hora Fim,Número Médio Ligantes,nan
0,1,00:00:00,00:15:00,601,14
1,2,00:15:00,00:30:00,190,7
2,3,00:30:00,00:45:00,180,8
3,4,00:45:00,01:00:00,177,7
4,5,01:00:00,01:15:00,162,6
...,...,...,...,...,...
91,92,22:45:00,23:00:00,920,23
92,93,23:00:00,23:15:00,897,21
93,94,23:15:00,23:30:00,571,15
94,95,23:30:00,23:45:00,560,14


In [108]:
df_tabela_ligantes.to_excel("df_nan.xlsx", index=False)